# Eksperimen Machine Learning - Heart Disease Dataset

**Nama:** I Kadek Rai Pramana  
**Dataset:** Heart Disease (UCI ML Repository)  
**Objective:** Klasifikasi binary apakah pasien memiliki penyakit jantung atau tidak  

---

## Outline
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Summary

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
        'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv(url, names=cols, na_values='?')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

Dataset shape: (303, 14)
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [2]:
print('First 5 rows:')
df.head()

First 5 rows:


age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target


In [3]:
print('Data Types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nDuplicates: {df.duplicated().sum()}')

Data Types:
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
target        int64
dtype: object

Missing values:
ca      4
thal    2
dtype: int64

Duplicates: 0


In [4]:
print('Statistical Summary:')
df.describe()

Statistical Summary:


       age  sex  cp  trestbps  chol  ...

## 2. Exploratory Data Analysis (EDA)

In [5]:
# Target distribution
print('Target Distribution (Original):')
print(df['target'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original target
df['target'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Original Target Distribution')
axes[0].set_xlabel('Target Class')
axes[0].set_ylabel('Count')

# Binary target
binary = (df['target'] >= 1).astype(int)
binary.value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'])
axes[1].set_title('Binary Target (0=Healthy, 1=Disease)')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

Target Distribution (Original):
0    164
1     55
2     36
3     35
4     13


In [6]:
# Distribution of numerical features
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=25, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [7]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Correlation Heatmap - Heart Disease Features', fontsize=14)
plt.tight_layout()
plt.show()

In [8]:
# Boxplot for outlier detection
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(f'{col}')

plt.suptitle('Boxplot - Outlier Detection', fontsize=14)
plt.tight_layout()
plt.show()

In [9]:
# Feature vs Target analysis
df_temp = df.copy()
df_temp['heart_disease'] = (df_temp['target'] >= 1).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for label, color in [(0, 'green'), (1, 'red')]:
        subset = df_temp[df_temp['heart_disease'] == label]
        axes[i].hist(subset[col].dropna(), bins=20, alpha=0.6, color=color,
                     label=f'{"Healthy" if label==0 else "Disease"}')
    axes[i].set_title(f'{col} by Heart Disease', fontsize=12)
    axes[i].legend()

axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [10]:
# Handle missing values
print('Missing values before:')
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in df.columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

print(f'\nMissing values after filling with median: {df.isnull().sum().sum()}')

Missing values before:
ca      4
thal    2

Missing values after filling with median: 0


In [11]:
# Outlier removal using IQR method
print(f'Rows before outlier removal: {len(df)}')

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print(f'Rows after outlier removal: {len(df)}')
print(f'Outliers removed: {303 - len(df)}')

Rows before outlier removal: 303
Rows after outlier removal: 284
Outliers removed: 19


In [12]:
# Create binary target
df['heart_disease'] = (df['target'] >= 1).astype(int)
df = df.drop('target', axis=1)

print('Binary target distribution:')
print(f'No Disease (0): {(df["heart_disease"] == 0).sum()}')
print(f'Has Disease (1): {(df["heart_disease"] == 1).sum()}')

Binary target distribution:
No Disease (0): 158
Has Disease (1): 126


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

X = df.drop('heart_disease', axis=1)
y = df['heart_disease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

Train set: 227 samples
Test set: 57 samples


In [14]:
# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns, index=X_test.index
)
print('Scaling applied.')

# Save scaler
import os
output_dir = 'heart_disease_preprocessing'
os.makedirs(output_dir, exist_ok=True)
joblib.dump(scaler, os.path.join(output_dir, 'scaler.pkl'))
print(f'Scaler saved to preprocessing/{output_dir}/scaler.pkl')

Scaling applied.
Scaler saved to preprocessing/heart_disease_preprocessing/scaler.pkl


In [15]:
# Save preprocessed data
train_df = X_train_scaled.copy()
train_df['heart_disease'] = y_train.values
test_df = X_test_scaled.copy()
test_df['heart_disease'] = y_test.values

train_df.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
test_df.to_csv(os.path.join(output_dir, 'test.csv'), index=False)

print('Files saved:')
print(f'  - {output_dir}/train.csv ({len(train_df)} rows)')
print(f'  - {output_dir}/test.csv ({len(test_df)} rows)')

Files saved:
  - heart_disease_preprocessing/train.csv (227 rows)
  - heart_disease_preprocessing/test.csv (57 rows)


In [16]:
# Verify saved data
verify = pd.read_csv(os.path.join(output_dir, 'train.csv'))
print(f'\nVerification - Train data:')
print(f'  Shape: {verify.shape}')
print(f'  Columns: {list(verify.columns)}')
print(f'  Target: {dict(verify["heart_disease"].value_counts())}')


Verification - Train data:
  Shape: (227, 14)
  Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'heart_disease']
  Target: {0: 127, 1: 100}


## 4. Summary

### Hasil Eksperimen:
- **Dataset:** Heart Disease dari UCI ML Repository (303 sampel, 14 fitur)
- **Missing Values:** 6 values (ca: 4, thal: 2) → diisi dengan median
- **Outliers:** 19 outlier dihapus menggunakan metode IQR
- **Target:** Multi-class (0-4) → Binary (0=No Disease, 1=Has Disease)
- **Split:** 80% train (227), 20% test (57) dengan stratified sampling
- **Scaling:** StandardScaler pada semua fitur
- **Output:** train.csv, test.csv, scaler.pkl